# 11 — Eight-class published benchmark: independent train/validation

**Purpose:** Create separate 8-class Urdu training and validation sets from the original headlines repository, excluding the published test examples.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


### Why this notebook matters

[The EMNLP 2024 paper](https://aclanthology.org/2024.findings-emnlp.426/) credits the original [Urdu-News-Headline-Dataset](https://github.com/mwaseemrandhawa/Urdu-News-Headline-Dataset), which provides Excel files for Business, Entertainment, Health, Politics, Science, Sports, Weird, and World. The published **1,000-line test CSV** is **not** training data. We remove exact matching test headlines **before** splitting any source training data. "Weird" is tentatively mapped to the paper's `other` category, but this mapping must be reviewed because the category names are not identical.

The dataset repository does not state a clear redistribution license; do not upload the original spreadsheets as part of your project without permission. Install `openpyxl` only to read the spreadsheets.

In [ ]:
import subprocess, pandas as pd
SOURCE_REPO = ROOT/'data/external/Urdu-News-Headline-Dataset'
PAPER_REPO  = ROOT/'data/external/Generalists-vs-Specialists'
CLONE_REPOSITORIES = False  # Set True after enabling Internet in Kaggle/Colab
if CLONE_REPOSITORIES:
    SOURCE_REPO.parent.mkdir(parents=True,exist_ok=True)
    for url,path in [('https://github.com/mwaseemrandhawa/Urdu-News-Headline-Dataset',SOURCE_REPO),
                     ('https://github.com/sameearif/Generalists-vs-Specialists',PAPER_REPO)]:
        if not path.exists(): subprocess.check_call(['git','clone','--depth','1',url,str(path)])
print('Original source files:',[p.name for p in SOURCE_REPO.glob('*.xlsx')])
print('Published test:',PAPER_REPO/'datasets/topic-classification.csv')

In [ ]:
# INSPECT before importing. Avoid guessing the headline column in the source XLSX files.
example_file=next(iter(SOURCE_REPO.glob('*.xlsx')),None)
if example_file:
    table=pd.read_excel(example_file,nrows=5)
    print('Example category file:',example_file.name)
    print('Available columns:',list(table.columns))
    display(table.head(3))
else: print('Clone source repository and ensure openpyxl is installed.')

### Explicit source-field mapping and safety gates

Set `HEADLINE_COLUMN` to the column containing the actual headline after inspecting the previous output. For example, it might be `Title`, but do **not** assume this is true. Set `CONFIRMED_CATEGORY_MAPPING=True` only after confirming that `Weird.xlsx` corresponds to the paper's `other` class.

In [ ]:
HEADLINE_COLUMN=None  # exact column name based on inspection above
CONFIRMED_CATEGORY_MAPPING=False  # review Weird -> other first
BUILD_DATA=False
MAX_TRAIN_PER_CLASS=500
MAX_VAL_PER_CLASS=50
SEED=42
CAT_FILES={'Business':'business','Entertainment':'entertainment','Health':'health',
           'Politics':'politics','Science':'science','Sports':'sports','Weird':'other','World':'world'}
print('Paper class order:',PAPER_LABELS)

In [ ]:
import re,unicodedata,random,hashlib
from collections import Counter

def fingerprint(text):
    text=unicodedata.normalize('NFKC',str(text)).casefold()
    text=re.sub(r'[​-‏﻿]','',text)
    return re.sub(r'\s+',' ',text).strip()

if BUILD_DATA:
    assert HEADLINE_COLUMN and CONFIRMED_CATEGORY_MAPPING, 'Review columns and class map before starting.'
    test_file=PAPER_REPO/'datasets/topic-classification.csv'
    assert test_file.exists(), 'Run notebook 08 or clone the paper repo'
    paper_test=read_csv(test_file)
    assert len(paper_test)==1000 and set(paper_test[0])>={'input','label'}
    test_fingerprints={fingerprint(r['input']) for r in paper_test}
    sources=[]
    for fname,label in CAT_FILES.items():
        file=SOURCE_REPO/f'{fname}.xlsx'
        assert file.exists(),f'Missing {file}'
        source=pd.read_excel(file)
        assert HEADLINE_COLUMN in source, f'{HEADLINE_COLUMN} missing in {file.name}, columns={list(source.columns)}'
        extracted=source[HEADLINE_COLUMN].dropna().astype(str).tolist()
        print(file.name,'rows:',len(extracted))
        for text in extracted:
            if len(fingerprint(text))>3: sources.append((fingerprint(text),text,label))
    # Reject duplicate source headlines across classes, not just within one class.
    by_fingerprint={}
    for fp,txt,label in sources: by_fingerprint.setdefault(fp,set()).add(label)
    ambiguous={fp for fp,labset in by_fingerprint.items() if len(labset)>1}
    unique={}
    for fp,txt,label in sources:
        if fp in test_fingerprints or fp in ambiguous: continue
        unique.setdefault(fp,(txt,label))
    print('Discarded test overlaps (unique):',len({fp for fp,_,_ in sources if fp in test_fingerprints}))
    print('Discarded same-headline conflicting classes:',len(ambiguous))
    grouped={label:[] for label in PAPER_LABELS}
    for fp,(txt,label) in unique.items(): grouped[label].append((fp,txt))
    train,val=[],[]
    for label in PAPER_LABELS:
        group=sorted(grouped[label]); random.Random(f'{SEED}:{label}').shuffle(group)
        assert len(group)>=MAX_TRAIN_PER_CLASS+MAX_VAL_PER_CLASS, f'Not enough safe {label} rows: {len(group)}'
        chosen_val=group[:MAX_VAL_PER_CLASS]
        chosen_train=group[MAX_VAL_PER_CLASS:MAX_VAL_PER_CLASS+MAX_TRAIN_PER_CLASS]
        for split,selected,target in [('train',chosen_train,train),('validation',chosen_val,val)]:
            for fp,txt in selected:
                target.append(dict(example_id=stable_id('paper8','ur',split,hashlib.sha256(fp.encode()).hexdigest()[:16]),
                                   index_id='',source='original headlines repo',lang='ur',split=split,
                                   text=txt,label=label,label_id=PAPER_LABELS.index(label)))
    test=[]
    for i,r in enumerate(paper_test):
        v=int(r['label'])
        test.append(dict(example_id=stable_id('paper8','ur','test',i),index_id='',
                         source='EMNLP2024 held-out evaluation',lang='ur',split='test',
                         text=r['input'],label=PAPER_LABELS[v],label_id=v))
    assert not validate_split(train+val+test,PAPER_LABELS)
    out=ROOT/'data/paper8';out.mkdir(parents=True,exist_ok=True)
    for name,rs in [('train',train),('validation',val),('test',test)]:
        write_csv(out/f'ur_{name}.csv',rs)
    write_json(out/'manifest.json',dict(task='paper8',labels=PAPER_LABELS,source_repository='mwaseemrandhawa/Urdu-News-Headline-Dataset',
               paper_test_sha256=file_hash(test_file),seed=SEED,
               notes='Source classes checked; exact text deduplicated; 8-class original paper test.'))
    print('READY for 03/04/05 with TASK=paper8 and counts:',len(train),len(val),len(test))
else:
    print('Data build disabled until you inspect the spreadsheet column and confirm the class mapping.')

**Remaining limitation:** Exact-match filtering does not prove that near-duplicate headlines or translated versions are absent. Inspect high-similarity train/test headlines before claiming no contamination. For a paper comparison, retain the exact 1,000 test inputs and the same macro-F1 calculation; training settings differ from the published 8B experiment and must be disclosed.